In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.21 Path-Integral Monte Carlo: Coin Flips Compute Quantum Mechanics

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.21",
    title="Path-Integral Monte Carlo: Coin Flips Compute Quantum Mechanics",
    blurb="Last notebook proved a thermal quantum particle is a classical ring polymer with "
    "honest positive weights; this notebook puts the license to work. Metropolis, the "
    "volume's oldest algorithm, walks the polymer, and quantum expectation values "
    "fall out of coin flips with error bars. We validate the sampler against exact "
    "finite-M targets before trusting it anywhere, watch local moves drown as the "
    "discretization refines and rescue them with exact Brownian bridges, separate "
    "statistical from Trotter error like adults, and then send the method where "
    "no formula can follow: an oscillator with no closed-form solution.",
    difficulty="advanced",
    estimate="210–250 min",
)

## Notebook overview

Movement VI's second notebook is the algorithmic consequence of its first. [§7.20](imaginary-time-quantum-classical.ipynb) ended with
a license: the thermal quantum particle *is* a classical ring polymer whose Boltzmann
weight $e^{-S}$ is **positive**, and positive weights are precisely what Metropolis ([§5.8](../05-classical-stat-mech/partition-function.ipynb),
the volume's oldest tool, invoked here with affection) knows how to sample. So quantum
mechanics at temperature becomes something a random walk can explore, and every quantum
expectation value falls out of coin flips. Nothing in this notebook's *physics* is new; that
is the point. What is new, and what the notebook actually spends its effort on, is
**epistemics: how does one come to trust a stochastic answer?**

The trust protocol is stated up front and followed to the letter. Rule one: validate the
sampler against an *exactly known* target before trusting it anywhere else, and [§7.20](imaginary-time-quantum-classical.ipynb) hands
us a whole family of them, because at every finite $M$ the harmonic polymer is pure Gaussian
algebra: $\langle x^2\rangle = (A^{-1})_{00}$ from `numpy.linalg.inv` and
$E_M = -\partial_\beta \ln Z_M$ from `numpy.linalg.slogdet`. Rule two: size every error bar
by the *measured* autocorrelation time, with a window whose stability is itself checked.
Rule three: keep the statistical budget and the Trotter budget separate, and report both.
The protocol is then dramatized by a true story from this notebook's own construction: a
first staging run appeared biased at $2.5\sigma$ against a target known to be exact; the
bridge, tested in isolation, was flawless; the culprit was a truncated autocorrelation
window that had quietly clipped a slow tail and shrunk the bars. The moral is the most
valuable sentence here: **when a validated-exact target disagrees with your sampler,
suspect the error bars before the physics.**

The arc: single-bead checkerboard Metropolis earns provisional trust at $M = 16$ (one bar
from the exact target, autocorrelation honestly quoted as the price of admission). Then the
**stiffening problem** is measured, not asserted: the springs grow as $M/\beta$, single-bead
steps shrink, and $\tau_{\mathrm{int}}$ climbs by a factor of six as $M$ goes $16 \to 64$,
local updates drowning near the continuum limit (the critical slowing of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb), one dimension
over). **Staging** is the cure and a small gem: the spring action is Gaussian, so a whole
segment can be redrawn *exactly* from its conditional distribution, a Brownian bridge built
bead by bead, with Metropolis accepting on the potential alone; acceptance then sits near
one *independent of* $M$. The deeper lesson arrives with it: the slow mode is the polymer's
**centroid**, and a bridge too short to displace it leaves the slowness in place. Two error
budgets are then separated on the $M$-ladder, each rung agreeing with *its own* exact
finite-$M$ value while the ladder converges onto the $\tfrac12\coth(\beta/2)$ of [§7.5](oscillator-at-temperature.ipynb): the coth
curve computed by ladder operators ([§7.5](oscillator-at-temperature.ipynb)), by Gaussian determinants ([§7.20](imaginary-time-quantum-classical.ipynb)), and now by coin
flips, the volume's tightest three-way rendezvous. The estimator lesson is measured on the
same runs (primitive and virial agree in expectation; their variances differ by a factor
that *grows* with $M$), and the method then proves its worth on the quartic oscillator,
where no closed form exists and grid ED supplies the experiment: agreement within one bar
on both energy and width, both budgets quoted. The horizons close Movement VI's technical
arc: bosonic exchange as loop reconnection, the sign problem named honestly, and ring-polymer
molecular dynamics as the bridge to production simulation.

> **Conventions (this notebook).** $\hbar = m = 1$ throughout; the harmonic studies run at
> $\omega = 1$, $\beta = 2$ unless stated. $\delta\tau = \beta/M$; ring closure
> $x_M \equiv x_0$. Every stochastic study seeds its own `numpy.random.default_rng` (seed
> stated in place). A single-bead **sweep** is one full checkerboard pass (every bead
> proposed once, evens then odds); a staging **sweep** is $M/L$ segment redraws. Metropolis
> exponents are clipped (`numpy.clip(dS, -50, 50)` for the vectorized mover; an
> accept-if-negative branch for staging) so no exponential ever overflows. Autocorrelation
> times come from `tau_int` (`numpy.correlate`, self-consistent window $W \ge c\,\tau$ with
> $c = 8$, stability checked against $c = 6, 10$); error bars from `blocked_error`
> (`numpy.array_split`, block length $16\,\tau$, the factor justified in Exercise 2).
> Exact finite-$M$ targets reuse the machinery of [§7.20](imaginary-time-quantum-classical.ipynb): `numpy.linalg.inv` of the circulant
> action matrix and `numpy.linalg.slogdet` for $\ln Z_M$. Equilibration cuts are visible
> in the traces and stated per study.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the sampler against the exact $(A^{-1})_{00}$; the window against its
> own stability; the stiffening against its measured growth; the bridge against the exact
> Brownian-bridge variances; every ladder rung against its own Gaussian value; two
> estimators against one exact energy; the quartic against grid ED. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** Bosonic permutation sampling, the sign problem (named, not developed), and
> ring-polymer molecular dynamics are outward horizons. See Ceperley, *Rev. Mod. Phys.*
> **67**, 279 (1995); Chandler & Wolynes 1981; Tuckerman, *Statistical Mechanics: Theory
> and Molecular Simulation* (staging and estimators); Sokal's Cargèse lectures (Monte
> Carlo error analysis). Cross-reference [§5.7](../05-classical-stat-mech/potentials-legendre-maxwell.ipynb) (Monte Carlo's
> dimension-blindness), [§5.8](../05-classical-stat-mech/partition-function.ipynb) (Metropolis and all its disciplines), [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) (critical slowing,
> cousin), [§7.5](oscillator-at-temperature.ipynb) (the coth, third derivation), [§7.17](bose-einstein-condensation.ipynb) (the condensate honesty met again),
> [§7.20](imaginary-time-quantum-classical.ipynb) (the license and the exact targets), and forward to [§7.22](eigenstate-thermalization.ipynb) (why isolated quantum
> systems thermalize at all, the volume's closing question).

## Theory in brief

### The license, put to work

[§7.20](imaginary-time-quantum-classical.ipynb) closed with the identity this notebook samples: inserting position states between
Trotter slices turns the thermal trace into the configuration integral of a classical ring
polymer, with nothing approximate anywhere in the construction (Chandler & Wolynes 1981;
Feynman's *Statistical Mechanics*, Ch. 3, carries it out in full). In the discrete notation
used throughout,

```{math}
:label: eq-pimc-license
Z_M \propto \int\!\prod_k dx_k\, e^{-S},
\qquad
S = \sum_{k=0}^{M-1}\left[\frac{M}{2\beta}(x_{k+1}-x_k)^2 + \frac{\beta}{M}V(x_k)\right],
\qquad x_M \equiv x_0,
```

and $e^{-S} > 0$ for every configuration. That single inequality is the notebook. For a
distinguishable particle (or bosons, where permutations *add* positive terms) the
ring-polymer weight is a genuine unnormalized probability, so the Metropolis algorithm of
[§5.8](../05-classical-stat-mech/partition-function.ipynb) applies verbatim, without one new idea: propose, compute $\Delta S$, accept with
probability $\min(1, e^{-\Delta S})$, and every quantum expectation becomes a classical
average over polymer shapes. (Fermions attach $(-1)^P$ to exchange permutations and the
weights stop being probabilities; that exception is flagged now and honored in the closing
paragraph.) The real subject is announced at the same time: a stochastic answer is only as
good as the argument for trusting it, and the argument here has three rules. **Validate on
an exactly known target first.** **Size the bars by the measured autocorrelation, with a
checked window.** **Separate the statistical budget from the Trotter budget.**

### Single-bead moves, checkerboard-vectorized

The simplest way to walk the polymer is the move [§5.8](../05-classical-stat-mech/partition-function.ipynb) would reach for first: displace one
bead, compute the change in action, accept or reject. Written out for bead $k$,

```{math}
:label: eq-single-bead
x_k \to x_k + \delta\,U(-1,1),
\qquad
\Delta S = \frac{M}{2\beta}\Big[(x_k'-x_{k-1})^2 + (x_k'-x_{k+1})^2 - \text{(old)}\Big]
+ \frac{\beta}{M}\big[V(x_k') - V(x_k)\big].
```

The action couples nearest neighbors only, so $\Delta S$ for bead $k$ involves $x_{k\pm1}$
and nothing else. On a ring with even $M$ that buys full vectorization: the even beads'
neighbors are all odd, so *conditioned on the odd beads the even ones are independent* and
may be updated simultaneously (and vice versa); one checkerboard pass is exactly equivalent
to a sequence of valid single-site updates. (The argument breaks the moment the springs
couple beyond nearest neighbors.) The step $\delta$ is tuned for reasonable acceptance and
both are reported. Validated below against the exact finite-$M$ target at $M = 16$,
$\beta = 2$, with the autocorrelation time honestly quoted as the price.

### Diagnostics: the 5.8 disciplines at full strength

A Metropolis chain hands back a *correlated* time series, so the error machinery of [§5.8](../05-classical-stat-mech/partition-function.ipynb)
(equilibration cuts, autocorrelation times, blocking) returns here at full strength; the
window analysis below follows the standard treatment in Sokal's Cargèse lectures, which
reward reading in the original. Two definitions carry everything:

```{math}
:label: eq-diagnostics
\tau_{\mathrm{int}} = \tfrac12 + \sum_{t=1}^{W}\rho(t),
\quad W \ge c\,\tau_{\mathrm{int}}(W),\ c = 8;
\qquad
\mathrm{err} = \frac{\sigma_{\text{blocks}}}{\sqrt{n_{\text{blocks}}}},
\quad \ell_{\text{block}} = 16\,\tau_{\mathrm{int}} .
```

Every stochastic number in this notebook carries a bar, the bar carries a
$\tau_{\mathrm{int}}$, and the $\tau_{\mathrm{int}}$ carries a window check. The
integrated autocorrelation time is computed from the normalized autocovariance
(`numpy.correlate`) summed out to a **self-consistent window**: the smallest $W$ with
$W \ge c\,\tau(W)$. The constant $c$ matters more than it looks, and this notebook knows it
the hard way: during construction, a $6\tau$ cutoff clipped a slow tail, the bars came out
too small, and honest staging data appeared biased at $2.5\sigma$ against a target known to
be exact. The bridge, tested in isolation, was flawless; the *bars* were the defect. Hence
the standing mandate: $c \ge 8$, **and** check that $\tau(c)$ is stable against $c = 6, 10$
before trusting it. Error bars come from blocking (`numpy.array_split`): block means are
nearly independent once blocks are much longer than $\tau$, and Exercise 2 measures the
residual creep to justify block length $16\tau$.

### The stiffening problem, measured

Why local moves must eventually fail is already visible in the action of Eq.
{eq}`eq-pimc-license`: the spring constant carries a factor $M/\beta$. The chain of
consequences is short,

```{math}
:label: eq-stiffening
k_{\text{spring}} = \frac{M}{\beta}
\quad\Longrightarrow\quad
\delta_{\text{acceptable}} \sim \sqrt{\beta/M}
\quad\Longrightarrow\quad
\tau_{\mathrm{int}} \ \text{climbs with } M .
```

Refining the discretization stiffens the springs linearly in $M$: each bead is chained ever
more tightly to its neighbors, acceptable single-bead steps shrink, and the polymer's
large-scale shape decorrelates ever more slowly. Measured below: at fixed $\delta$ the
autocorrelation time grows by a factor of six from $M = 16$ to $M = 64$ while acceptance
falls. This is the generic fate of local updates near a continuum limit, and the critical
slowing of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) is its cousin: there the correlation *length* diverged, here the spring stiffness
does, and in both cases local moves stop carrying information across the system.

### Staging: exact Brownian bridges

The cure exploits the one special structure the action has: its spring part is Gaussian,
and conditionals of a Gaussian are again Gaussian, with means and variances one can write
down. That permits *exact* resampling of a whole segment, a construction known as staging
(Tuckerman's *Statistical Mechanics: Theory and Molecular Simulation* develops it in full).
The working formula is the one-bead conditional,

```{math}
:label: eq-staging
x_i \,\big|\, x_{i-1}, x_R
\ \sim\ \mathcal{N}\!\left(\frac{r\,x_{i-1} + x_R}{r+1},\ \delta\tau\,\frac{r}{r+1}\right),
\qquad r = \text{links remaining to } x_R .
```

where the derivation is precision weighting, in two lines: bead $x_i$ feels its already-placed left neighbor
through one link (precision $1/\delta\tau$) and the far endpoint $x_R$ through the $r$
links that remain (an effective spring of precision $1/r\delta\tau$); multiplying the two
Gaussians gives mean $(r\,x_{i-1} + x_R)/(r+1)$ and variance $\delta\tau\,r/(r+1)$. Built
bead by bead this is the Brownian bridge, and because the springs are sampled exactly,
Metropolis accepts on the *potential alone*: acceptance sits near one independent of $M$.
The bridge is verified in isolation below (interior means on the straight line between
endpoints; variances against the exact $\delta\tau\,k(L-k)/L$), which is precisely the test
that separated correctness from statistics in the cautionary tale. And the diagnosis
exposes which mode was slow all along: the polymer's **centroid**, which a short bridge
barely displaces. Segment length matters ($L = M/4$ versus $M/2$ is measured below, a
factor of three in $\tau$), and production codes add an explicit whole-ring displacement
move ($x_k \to x_k + \Delta$ for all $k$, springs untouched) precisely to carry the
centroid; here sizing $L$ to $M/2$ suffices. The design rule: smart moves are exact where
the action is Gaussian and humble where it is not.

### Two budgets

A PIMC number carries two entirely different errors, and the protocol's third rule is that
they are never conflated. Schematically,

```{math}
:label: eq-two-budgets
\underbrace{\big|\langle x^2\rangle_{\text{PIMC}} - (A^{-1})_{00}\big| \lesssim \text{bars}}_{\text{statistical}},
\qquad
\underbrace{(A^{-1})_{00} \xrightarrow{\ 1/M^2\ } \tfrac12\coth(\beta/2)}_{\text{Trotter}} .
```

The **statistical** error is the blocking bar, sized by $\tau_{\mathrm{int}}$; it shrinks as
$1/\sqrt{N}$ and belongs to the sampler. The **Trotter** error is the distance from the
finite-$M$ Gaussian value to the $M \to \infty$ limit; it falls as $1/M^2$ (the trace
theorem of [§7.20](imaginary-time-quantum-classical.ipynb): partition functions get second-order accuracy free) and belongs to the
discretization, not the sampler. The $M$-ladder below exhibits both at once: each rung's
sampled mean agrees with *its own* exact $(A^{-1})_{00}$ within bars, while the rungs
themselves converge onto the $\tfrac12\coth(\beta/2)$ of [§7.5](oscillator-at-temperature.ipynb). The coth curve, by ladder
operators in [§7.5](oscillator-at-temperature.ipynb), by Gaussian determinants in [§7.20](imaginary-time-quantum-classical.ipynb), and now by coin flips: the volume's
tightest three-way rendezvous, this time wearing error bars.

### Primitive versus virial

Nothing singles out one formula for the energy: any function of the beads whose average
reproduces the thermodynamic derivative is a legitimate estimator, and differentiating
$\ln Z_M$ along two different routes yields two standard, inequivalent ones (both
derivations appear in Exercise 6; Tuckerman treats the general case):

```{math}
:label: eq-estimators
E_{\text{prim}} = \frac{M}{2\beta} - \frac{M}{2\beta^2}\sum_k\langle(\Delta x_k)^2\rangle
+ \Big\langle \overline{V} \Big\rangle,
\qquad
E_{\text{vir}} = \Big\langle \overline{\tfrac12 x V'(x) + V(x)} \Big\rangle,
```

with the bar denoting a bead average. Both are exact at finite $M$ and target the *same*
number $-\partial_\beta \ln Z_M$: the primitive comes from differentiating $\ln Z_M$
directly (the $\beta$-dependence of prefactor, springs, and potential), the virial from
rescaling $x \to \sqrt{\beta}\,u$ so that $\beta$ survives only inside the potential, then
differentiating. Equal in expectation, they are *not* equal in practice: the primitive is
the difference of two terms that each grow linearly with $M$ (the $M/2\beta$ constant and
the spring sum), and while the means cancel to $O(1)$, the *fluctuations do not*, so its
per-sample standard deviation grows as $\sqrt{M}$. The virial never forms large canceling
terms and its variance stays flat. Both behaviours are measured below, and the standing
rule issued: virial for production, primitive for cross-checks.

### Where no formula exists

The certification study needs two ingredients chosen with care: a target with no
closed-form solution, and an independent numerical experiment trusted enough to referee.
Both fit in one line,

```{math}
:label: eq-quartic
V(x) = \tfrac12 x^2 + \tfrac12 x^4,
\qquad \beta = 5 :
\qquad
\text{grid ED (Vol VI)} \ \leftrightarrow\ \text{PIMC with the full protocol}.
```

The harmonic oscillator validated the machinery; it cannot certify the method, because for
it we never needed Monte Carlo at all. The quartic oscillator has no closed form, so the
"experiment" is supplied by grid exact diagonalization (the three-point Laplacian and
`numpy.linalg.eigh`, Volume VI's machinery in one reverent line; box and spacing checked
against the thermal width). PIMC at $M = 64$ with the full protocol then delivers energy
and width with honest bars, both budgets quoted. And then the sentence that justifies the
whole enterprise: the grid dies exponentially with dimension (this resolution is 1500
points in 1D, $3\times10^9$ in 3D, hopeless for $N$ particles), while the polymer's cost
grows polynomially. Monte Carlo's dimension-blindness ([§5.7](../05-classical-stat-mech/potentials-legendre-maxwell.ipynb)), quantum edition; this is why
liquid helium, not a toy oscillator, is the method's natural prey.

### Horizons

Bosonic **exchange** enters as loop reconnection: permutations of identical particles
connect $N$ small loops into fewer, longer ones (the braiding flag of [§7.20](imaginary-time-quantum-classical.ipynb)), and sampling those
reconnections alongside the bead moves is full bosonic PIMC. Ceperley's simulations of
liquid helium-4 (*Rev. Mod. Phys.* **67**, 279 (1995)) computed the superfluid fraction
and a condensate fraction $\langle N_0\rangle \approx 7$–$10\%$ this way, meeting the
neutron-scattering honesty of [§7.17](bose-einstein-condensation.ipynb) from the theory side. The **sign problem** is the boundary:
fermionic permutations carry $(-1)^P$, the weights stop being probabilities, signed
averages cancel almost completely, and the variance of the ratio explodes exponentially in
$\beta N$. No general cure is known; that is a statement of the field's honest frontier,
not a challenge dodged. And **ring-polymer molecular dynamics** runs this same polymer
with forces instead of coin flips: nuclear quantum effects in production molecular
simulation, the bridge to the MMM course, in one line.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: hbar = m = 1; harmonic studies at omega = 1, beta = 2 unless
# stated; dtau = beta/M; ring closure x_M = x_0. Each study seeds its own
# default_rng (seed stated in place) so every number below is reproducible
# in isolation. Sweep definitions: single-bead sweep = one full checkerboard
# pass (evens then odds, every bead proposed once); staging sweep = M/L
# segment redraws at uniformly random anchors.


def ring_matrix(M, beta, omega=1.0):
    """The ring polymer's Gaussian action matrix A (the helper of §7.20, restated).

    Action (hbar = m = 1): S = (1/2) x^T A x with
    A = (M/beta)(2I - S - S^T) + (beta omega^2/M) I, built from a numpy.roll
    circulant. The springs stiffen as M/beta — Exercise 3's villain.

    Parameters
    ----------
    M : int
        Number of beads.
    beta : float
        Inverse temperature.
    omega : float, optional
        Oscillator frequency (default 1).

    Returns
    -------
    numpy.ndarray
        The M × M circulant action matrix.
    """
    S = np.roll(np.eye(M), 1, axis=1)
    return (M / beta) * (2.0 * np.eye(M) - S - S.T) + (beta * omega**2 / M) * np.eye(M)


def ring_lnZ(M, beta, omega=1.0):
    """ln Z_M for the harmonic ring polymer, via numpy.linalg.slogdet only.

    ln Z_M = (M/2) ln(M/beta) - (1/2) ln det A. The raw determinant overflows
    float64 near M ~ 300 (§7.20 demonstrated this once); in logs the geometric
    factors cancel politely.

    Parameters
    ----------
    M : int
        Beads.
    beta : float
        Inverse temperature.
    omega : float, optional
        Frequency (default 1).

    Returns
    -------
    float
        ln Z_M.
    """
    sign, logdet = np.linalg.slogdet(ring_matrix(M, beta, omega))
    return float(0.5 * M * np.log(M / beta) - 0.5 * logdet)


def exact_x2_M(M, beta):
    """The exact finite-M bead variance <x^2> = (A^{-1})_00 (numpy.linalg.inv).

    This is Rule One's target: at every finite M the harmonic polymer is pure
    Gaussian algebra, so the sampler can be validated against a number that is
    exact BEFORE it is trusted anywhere else. Converges to coth(beta/2)/2 as
    M grows (the curve of §7.5).

    Parameters
    ----------
    M : int
        Beads.
    beta : float
        Inverse temperature.

    Returns
    -------
    float
        The exact finite-M thermal width (A^{-1})_00.
    """
    return float(np.linalg.inv(ring_matrix(M, beta))[0, 0])


def exact_E_M(M, beta, step=1e-5):
    """The exact finite-M energy E_M = -d(ln Z_M)/d(beta), central difference.

    Differentiates ring_lnZ numerically with the step stated (1e-5: small
    enough that the O(step^2) truncation sits far below the digits quoted,
    large enough to stay clear of round-off). Both estimators of Exercise 6
    target THIS number. For the harmonic oscillator E_M equals the finite-M
    virial combination <x^2>_M exactly — a cross-check gated in Exercise 6.

    Parameters
    ----------
    M : int
        Beads.
    beta : float
        Inverse temperature.
    step : float, optional
        Central-difference step in beta (default 1e-5).

    Returns
    -------
    float
        E_M, converging to (1/2) coth(beta/2) as M grows.
    """
    return float(-(ring_lnZ(M, beta + step) - ring_lnZ(M, beta - step)) / (2 * step))


def tau_int(series, c=8.0):
    """Integrated autocorrelation time with a self-consistent window.

    Computes the normalized autocovariance rho(t) via numpy.correlate, then
    tau(W) = 1/2 + sum_{t=1..W} rho(t), and returns tau at the SMALLEST window
    W with W >= c*tau(W) (Sokal's self-consistent criterion). The constant c
    is this notebook's scar tissue: a c = 6 window once clipped a slow tail,
    shrank the bars, and manufactured a phantom 2.5 sigma bias. Mandate: c >= 8,
    and check stability against c = 6 and 10 before trusting (Exercise 2).

    Parameters
    ----------
    series : array_like
        Time series of a scalar observable (one value per sweep).
    c : float, optional
        Window constant (default 8).

    Returns
    -------
    tau : float
        Integrated autocorrelation time, in sweeps.
    W : int
        The window actually used.
    """
    x = np.asarray(series, float)
    x = x - x.mean()
    n = x.size
    acf = np.correlate(x, x, mode="full")[n - 1 :]
    acf = acf / acf[0]
    taus = 0.5 + np.cumsum(acf[1 : n // 2])
    W_arr = np.arange(1, taus.size + 1)
    ok = W_arr >= c * taus
    if not ok.any():
        return float(taus[-1]), int(W_arr[-1])
    W = int(W_arr[np.argmax(ok)])
    return float(taus[W - 1]), W


def blocked_error(series, tau):
    """Blocking error bar with block length 16*tau (numpy.array_split).

    Block means become independent only when blocks are MUCH longer than tau;
    Exercise 2 measures the residual creep of the bar with block length and
    finds it stabilizes near 16*tau, which is therefore the standing choice.
    The bar is std(block means)/sqrt(n_blocks).

    Parameters
    ----------
    series : array_like
        Time series (post-equilibration).
    tau : float
        Integrated autocorrelation time of the series, from tau_int.

    Returns
    -------
    err : float
        The one-sigma error bar on the series mean.
    n_blocks : int
        Number of blocks used.
    """
    x = np.asarray(series, float)
    ell = max(1, int(np.ceil(16.0 * tau)))
    nblocks = max(5, x.size // ell)
    blocks = [
        b.mean() for b in np.array_split(x[: (x.size // nblocks) * nblocks], nblocks)
    ]
    return float(np.std(blocks, ddof=1) / np.sqrt(nblocks)), int(nblocks)


def V_harm(x):
    """Harmonic potential V = x^2/2 (omega = 1)."""
    return 0.5 * x * x


def dV_harm(x):
    """Harmonic force term V'(x) = x."""
    return x


def V_quart(x):
    """Quartic oscillator V = x^2/2 + x^4/2 — Exercise 7's closed-form-free target."""
    return 0.5 * x * x + 0.5 * x**4


def dV_quart(x):
    """Quartic derivative V'(x) = x + 2 x^3."""
    return x + 2.0 * x**3


def pimc_single_bead(M, beta, V, dV, nsweep, delta, rng, snap_every=0):
    """Checkerboard-vectorized single-bead Metropolis on the ring polymer.

    One sweep proposes x_k -> x_k + delta*U(-1,1) for every bead, evens then
    odds. The parallel update is valid because the action couples nearest
    neighbors only: with M even, same-parity beads are conditionally
    independent given the other parity, so the simultaneous update equals a
    sequence of legal single-site updates (and would NOT for longer-range
    springs). Metropolis exponents are clipped to [-50, 50] before
    exponentiation so nothing overflows.

    Records, per sweep: the bead-averaged width x2 = mean(x^2), the primitive
    energy estimator M/2beta - (M/2beta^2) sum (dx)^2 + mean(V), and the
    virial estimator mean(x V'(x)/2 + V(x)).

    Parameters
    ----------
    M : int
        Beads (must be even for the checkerboard argument).
    beta : float
        Inverse temperature.
    V, dV : callable
        Potential and its derivative, vectorized.
    nsweep : int
        Number of sweeps.
    delta : float
        Proposal half-width.
    rng : numpy.random.Generator
        Seeded generator (seed stated per study).
    snap_every : int, optional
        If positive, store a configuration snapshot every snap_every sweeps.

    Returns
    -------
    dict
        Keys x2, eprim, evir (per-sweep series), acc (acceptance fraction),
        x (final configuration), snaps (list of snapshots).
    """
    dt = beta / M
    k2 = M / (2.0 * beta)
    x = np.zeros(M)
    idx_e, idx_o = np.arange(0, M, 2), np.arange(1, M, 2)
    x2 = np.empty(nsweep)
    eprim = np.empty(nsweep)
    evir = np.empty(nsweep)
    n_acc = 0
    snaps = []
    for s in range(nsweep):
        for idx in (idx_e, idx_o):
            xl, xr = x[(idx - 1) % M], x[(idx + 1) % M]
            xo = x[idx]
            xn = xo + delta * rng.uniform(-1.0, 1.0, idx.size)
            dS = k2 * (
                (xn - xl) ** 2 + (xn - xr) ** 2 - (xo - xl) ** 2 - (xo - xr) ** 2
            )
            dS += dt * (V(xn) - V(xo))
            acc = rng.random(idx.size) < np.exp(-np.clip(dS, -50.0, 50.0))
            x[idx] = np.where(acc, xn, xo)
            n_acc += int(acc.sum())
        spr = float(np.sum((np.roll(x, -1) - x) ** 2))
        x2[s] = float(np.mean(x * x))
        eprim[s] = M / (2 * beta) - (M / (2 * beta**2)) * spr + float(np.mean(V(x)))
        evir[s] = float(np.mean(0.5 * x * dV(x) + V(x)))
        if snap_every and s % snap_every == 0:
            snaps.append(x.copy())
    return dict(
        x2=x2, eprim=eprim, evir=evir, acc=n_acc / (nsweep * M), x=x, snaps=snaps
    )


def staging_move(x, j, L, dt, V, rng):
    """One staging move: redraw L-1 interior beads by an exact Brownian bridge.

    Endpoints x_j and x_{j+L} stay fixed. Interior beads are built one at a
    time from the conditional distribution derived by precision weighting:
    with r links remaining to the right endpoint, bead i is Gaussian with
    mean (r*x_prev + x_R)/(r+1) and variance dtau*r/(r+1). Because the spring
    action is sampled exactly, Metropolis accepts on the POTENTIAL alone
    (accept outright if dS <= 0; otherwise compare to exp(-dS) with dS capped
    at 50 so the exponential never overflows).

    Parameters
    ----------
    x : numpy.ndarray
        Configuration, modified in place on acceptance.
    j : int
        Anchor bead (left endpoint of the segment).
    L : int
        Segment length in links; L-1 interior beads are redrawn.
    dt : float
        Imaginary-time step dtau = beta/M.
    V : callable
        Potential, vectorized.
    rng : numpy.random.Generator
        Seeded generator.

    Returns
    -------
    bool
        True if the move was accepted.
    """
    M = x.size
    idxs = (j + np.arange(1, L)) % M
    xL, xR = x[j], x[(j + L) % M]
    old = x[idxs]
    xi = rng.standard_normal(L - 1)
    new = np.empty(L - 1)
    prev = xL
    for i in range(1, L):
        r = L - i
        mean = (r * prev + xR) / (r + 1)
        var = dt * r / (r + 1)
        prev = mean + np.sqrt(var) * xi[i - 1]
        new[i - 1] = prev
    dS = dt * (float(np.sum(V(new))) - float(np.sum(V(old))))
    u = rng.random()
    if dS <= 0.0 or u < np.exp(-min(dS, 50.0)):
        x[idxs] = new
        return True
    return False


def pimc_staging(M, beta, V, dV, nsweep, L, rng, snap_every=0):
    """Staging PIMC: one sweep = M/L bridge redraws at random anchors.

    Segment length L is a physical choice, not a tuning detail: the slow mode
    is the polymer's centroid, and a bridge must be long enough to displace
    it (Exercise 4 measures L = M/4 versus M/2). Records the same per-sweep
    series as pimc_single_bead.

    Parameters
    ----------
    M : int
        Beads.
    beta : float
        Inverse temperature.
    V, dV : callable
        Potential and derivative, vectorized.
    nsweep : int
        Number of sweeps.
    L : int
        Bridge length in links (L-1 interior beads per move).
    rng : numpy.random.Generator
        Seeded generator (seed stated per study).
    snap_every : int, optional
        If positive, store a configuration snapshot every snap_every sweeps.

    Returns
    -------
    dict
        Keys x2, eprim, evir, acc, x, snaps as in pimc_single_bead.
    """
    dt = beta / M
    x = np.zeros(M)
    mps = max(1, M // L)
    x2 = np.empty(nsweep)
    eprim = np.empty(nsweep)
    evir = np.empty(nsweep)
    n_acc = 0
    snaps = []
    for s in range(nsweep):
        for _ in range(mps):
            j = int(rng.integers(M))
            n_acc += staging_move(x, j, L, dt, V, rng)
        spr = float(np.sum((np.roll(x, -1) - x) ** 2))
        x2[s] = float(np.mean(x * x))
        eprim[s] = M / (2 * beta) - (M / (2 * beta**2)) * spr + float(np.mean(V(x)))
        evir[s] = float(np.mean(0.5 * x * dV(x) + V(x)))
        if snap_every and s % snap_every == 0:
            snaps.append(x.copy())
    return dict(
        x2=x2, eprim=eprim, evir=evir, acc=n_acc / (nsweep * mps), x=x, snaps=snaps
    )


def grid_ed(Vfun, half_box, N):
    """Grid exact diagonalization: three-point Laplacian + numpy.linalg.eigh.

    Volume VI's machinery in one function: H = -(1/2) d^2/dx^2 + V on a
    uniform grid of N points spanning [-half_box, half_box], the Laplacian
    discretized to second order. Supplies the independent 'experiment' for
    Exercise 7; box and spacing adequacy are checked in place against the
    thermal width (the nmax discipline of §7.9, once more).

    Parameters
    ----------
    Vfun : callable
        Potential, vectorized.
    half_box : float
        Half-width of the box.
    N : int
        Number of grid points.

    Returns
    -------
    xg : numpy.ndarray
        Grid.
    E : numpy.ndarray
        Eigenvalues, ascending.
    U : numpy.ndarray
        Eigenvectors as columns (grid-normalized: sum U^2 = 1).
    """
    xg = np.linspace(-half_box, half_box, N)
    dx = xg[1] - xg[0]
    off = np.full(N - 1, 1.0)
    lap = (np.diag(off, 1) + np.diag(off, -1) - 2.0 * np.eye(N)) / dx**2
    E, U = np.linalg.eigh(-0.5 * lap + np.diag(Vfun(xg)))
    return xg, E, U


def ed_thermal(xg, E, U, beta, nkeep=80):
    """Thermal E(beta) and <x^2>(beta) from grid-ED eigenpairs.

    Boltzmann weights are ground-shifted (e^{-beta(E_n - E_0)}) so nothing
    underflows, truncated at nkeep states with the truncation checked by the
    weight of the last state kept.

    Parameters
    ----------
    xg, E, U : numpy.ndarray
        Output of grid_ed.
    beta : float
        Inverse temperature.
    nkeep : int, optional
        Number of eigenstates kept (default 80).

    Returns
    -------
    Eth : float
        Thermal energy.
    x2th : float
        Thermal width <x^2>.
    """
    w = np.exp(-beta * (E[:nkeep] - E[0]))
    w = w / w.sum()
    Eth = float(np.sum(w * E[:nkeep]))
    x2th = float(np.sum(w * ((U[:, :nkeep] ** 2).T @ (xg**2))))
    return Eth, x2th


BETA = 2.0
print("exact finite-M targets at beta = 2 (Rule One's family):")
for M_t in (8, 16, 32, 64):
    print(
        f"  M = {M_t:2d}:  <x2>_M = {exact_x2_M(M_t, BETA):.6f}   "
        f"E_M = {exact_E_M(M_t, BETA):.6f}"
    )
print(f"  continuum (the coth of §7.5): {0.5 / np.tanh(BETA / 2):.6f}")

## Exercise 1 — The license, put to work; the protocol declared

Metropolis meets the polymer, and the rules of trust are stated before any number is
believed. Cite {eq}`eq-pimc-license`, {eq}`eq-single-bead`.

1. Restate the polymer weight ([§7.20](imaginary-time-quantum-classical.ipynb)) and argue its positivity (the boson/distinguishable
   license, with the fermionic exception flagged for the close); state the three-rule
   protocol in prose.
2. Implement checkerboard single-bead Metropolis in `pimc_single_bead`
   (`numpy.random.default_rng(7211)`; the even/odd validity argument in one line; the step
   $\delta$ tuned and the acceptance reported).
3. Validate against the exact finite-$M$ target: the sampled $\langle x^2\rangle$ with its
   blocking bar against $(A^{-1})_{00}$ from `exact_x2_M(16, 2.0)`, at $M = 16$,
   $\beta = 2$; the sampler earns provisional trust.
4. Report the price honestly (the measured $\tau_{\mathrm{int}}$ in sweeps, from
   `tau_int`) and pose the question Exercise 3 answers: what happens to this price as
   $M$ grows?

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    x2_pimc1,
    x2_exact1,
    "first trust: the sampler meets an exact target within three bars",
    atol=3 * bar1,
)
validate.check(
    0.3 < run1["acc"] < 0.9,
    "the tuned mover proposes usefully (acceptance neither frozen nor diffusive)",
    f"acceptance {run1['acc']:.3f} at delta = {DELTA1}",
)

```{admonition} With your assistant
:class: tip
The trust protocol of Exercise 1 extends verbatim to code you did not write:
if your assistant generates a path sampler — a different move set, a different
update order — it enters this notebook the same way ours did, by reproducing
the exact finite-$M$ answer on the harmonic oscillator before it is allowed
near any system without one. Whoever wrote the sampler, the certification
step is not optional, and the check is yours.
```

## Exercise 2 — Error bars or it didn't happen

The disciplines of [§5.8](../05-classical-stat-mech/partition-function.ipynb) at full strength, including the cautionary tale this notebook's own
construction supplied. Cite {eq}`eq-diagnostics`.

1. Show the $\langle x^2\rangle$ trace from the cold start; mark and apply the
   equilibration cut.
2. Demonstrate `tau_int`'s self-consistent window with a stability check
   ($\tau(c)$ for $c = 6, 8, 10$ compared before trusting) and justify `blocked_error`'s
   block length by measuring the bar against block lengths $4\tau, 8\tau, 16\tau, 32\tau$
   (`numpy.array_split`).
3. Tell the true story: a $6\tau$ window clipped a slow tail, honest staging data looked
   $2.5\sigma$ biased, the bridge tested exact in isolation, and the bars were the
   culprit; state the moral.
4. Issue the reporting rule (prose): every stochastic number in this course carries a bar,
   the bar carries a $\tau$, and the $\tau$ carries a window check, no exceptions.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    tau_spread < 1.3 and 0.8 < bar_consistency < 1.25,
    "bars that can be trusted: window stable in c, blocking bar stable past 16 tau",
    f"tau spread {tau_spread:.2f}x, bar(32tau)/bar(16tau) = {bar_consistency:.2f}",
)

## Exercise 3 — The stiffening problem, measured

Local moves drown as the continuum approaches. Cite {eq}`eq-stiffening`.

1. Run `pimc_single_bead` at $M = 16$ and $M = 64$ at the *same* step $\delta = 0.5$
   (tuned once at $M = 16$; holding it fixed isolates the $M$-dependence, and the falling
   acceptance is itself part of the diagnosis) and measure $\tau_{\mathrm{int}}$ for both
   with `tau_int`.
2. Diagnose (prose plus one formula): springs stiffen as $M/\beta$, acceptable steps
   shrink as $\sqrt{\beta/M}$, and each bead's random walk slows accordingly.
3. Connect in one breath: critical slowing's cousin ([§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb)), local updates against a
   diverging stiffness; the generic fate stated.
4. Pose the cure's requirement: a move that respects the springs exactly; the next
   exercise builds it.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    stiff_factor > 3.0 and accs_stiff[64] < accs_stiff[16],
    "the stiffening problem, quantified: tau climbs strongly with M for local updates",
    f"tau {taus_stiff[16]:.0f} -> {taus_stiff[64]:.0f} sweeps ({stiff_factor:.1f}x), "
    f"acceptance {accs_stiff[16]:.2f} -> {accs_stiff[64]:.2f}",
)

## Exercise 4 — Staging: exact Brownian bridges (centerpiece)

Sample the springs exactly, accept on the potential, and learn which mode was actually
slow. Cite {eq}`eq-staging`.

1. Derive the bridge conditionals by precision weighting (two lines in the solution):
   mean $(r\,x_{\text{prev}} + x_R)/(r+1)$, variance $\delta\tau\,r/(r+1)$; they are
   implemented in `staging_move`.
2. Verify the bridge in isolation (the diagnosis's decisive test): interior means against
   the straight line between endpoints and variances against the exact
   $\delta\tau\,k(L-k)/L$, from $4\times10^5$ bridge draws
   (`numpy.random.default_rng(7214)`).
3. Measure the cure with `pimc_staging` at $M = 64$: acceptance near one, and the
   centroid lesson — $\tau_{\mathrm{int}}$ at $L = M/4$ versus $L = M/2$
   (`numpy.random.default_rng(7215)` for each run; the whole-ring displacement move named
   as the standard alternative).
4. Re-validate against the exact finite-$M$ target with honest bars and state the design
   rule (prose): smart moves are exact where the action is Gaussian and humble where it
   is not.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    np.array(vars_meas),
    vars_th,
    "the bridge, exact in isolation: variances on dtau*k(L-k)/L",
    rtol=3e-2,
)
validate.check(
    gap_mean < 3e-3,
    "and its means on the straight line between endpoints",
    f"max gap {gap_mean:.1e}",
)
validate.close(
    x2_stag,
    x2_ex64,
    "staging trusted, centroid tamed: exact target met with honest bars",
    atol=3 * bar_stag,
)
validate.check(
    stag[16]["acc"] > 0.9 and stag[32]["acc"] > 0.85 and centroid_factor > 2.0,
    "acceptance near one at both L; the centroid lesson measured",
    f"acc {stag[16]['acc']:.2f}/{stag[32]['acc']:.2f}, tau ratio {centroid_factor:.1f}x",
)

## Exercise 5 — Two budgets, separated

The $M$-ladder with bars: each rung checked against its own exact value, the ladder
converging on [§7.5](oscillator-at-temperature.ipynb). Cite {eq}`eq-two-budgets`.

1. Run the ladder $M = 8, 16, 32, 64$ with `pimc_staging` at $L = M/2$
   (`numpy.random.default_rng(7100 + M)` per rung) and compare each rung's
   $\langle x^2\rangle$ with blocking bars against its exact `exact_x2_M(M, 2.0)`.
2. Exhibit the two budgets: statistical (the bars) versus Trotter (the rung-to-limit
   distances, falling as $1/M^2$, the trace theorem of [§7.20](imaginary-time-quantum-classical.ipynb) recalled in one line); estimate
   each share at $M = 64$.
3. Say the rendezvous aloud (prose): $\tfrac12\coth(\beta/2)$ computed by ladder
   operators ([§7.5](oscillator-at-temperature.ipynb)), by Gaussian determinants ([§7.20](imaginary-time-quantum-classical.ipynb)), and now by coin flips.
4. Issue the reporting rule (prose): stochastic-plus-discretized results quote both
   budgets, always separately; conflated budgets are how wrong numbers acquire
   confidence.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
for M_l in (8, 16, 32, 64):
    d = ladder[M_l]
    validate.close(
        d["mean"],
        d["exact"],
        f"rung M = {M_l} agrees with its own exact finite-M value",
        atol=3 * d["bar"],
    )
validate.close(
    trotter_ratios,
    [4.0, 4.0, 4.0],
    "and the Trotter distances fall as 1/M^2 (the trace theorem of §7.20)",
    rtol=8e-2,
)

## Exercise 6 — (STUDENT) Primitive versus virial

Two unbiased estimators, one pathology, measured. Cite {eq}`eq-estimators`.

1. Derive both estimators (the primitive by $-\partial_\beta \ln Z_M$; the virial by the
   rescaling $x \to \sqrt{\beta}\,u$ and then $-\partial_\beta$); both are recorded per
   sweep by `pimc_staging` as `eprim` and `evir`.
2. Verify both against the exact finite-$M$ energy from `exact_E_M(M, 2.0)` (the central
   difference of the `slogdet` formula of [§7.20](imaginary-time-quantum-classical.ipynb), step $10^{-5}$), on the Exercise 5 runs.
3. Measure the pathology: the per-sample standard deviation of $E_{\text{prim}}$ against
   $M = 8 \to 64$ (`numpy.std`) versus the flat $E_{\text{vir}}$; explain (two $O(M)$
   terms canceling in the mean but not in the fluctuations).
4. Issue the rule (prose): virial for production, primitive for cross-checks; estimators
   equal in expectation are not equal in practice, and choosing observables is part of
   the method.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
for M_l in (8, 64):
    mp, bp, mv, bv, E_ex = est_rows[M_l]
    validate.close(
        mp,
        E_ex,
        f"the primitive estimator meets the exact target at M = {M_l}",
        atol=3 * bp,
    )
    validate.close(
        mv,
        E_ex,
        f"and the virial estimator meets the same target at M = {M_l}",
        atol=3 * bv,
    )
validate.check(
    prim_growth > 2.0 and vir_growth < 1.5,
    "the variance pathology, measured: primitive grows with M, virial stays flat",
    f"sd_prim x{prim_growth:.1f}, sd_vir x{vir_growth:.1f} across M = 8 -> 64",
)
validate.check(
    energy_width_gap < 1e-8,
    "harmonic cross-check: exact finite-M energy equals finite-M width (virial identity)",
    f"gap {energy_width_gap:.1e}",
)

## Exercise 7 — Where no formula exists (centerpiece)

An oscillator with no closed form: PIMC versus grid ED, with the full protocol. Cite
{eq}`eq-quartic`.

1. Build the grid-ED experiment for $V = x^2/2 + x^4/2$ with `grid_ed` (three-point
   Laplacian, `numpy.linalg.eigh`, 1500 points on $[-4, 4]$; box and spacing adequacy
   checked against the thermal width) and compute $E_0$, $E(\beta{=}5)$, and
   $\langle x^2\rangle$ from `ed_thermal`.
2. Run PIMC at $M = 64$ with the full protocol (`numpy.random.default_rng(7217)`, stated
   cut, window-checked $\tau$ from `tau_int`, blocking from `blocked_error`, the virial
   estimator) and compare both observables; quote both budgets, the Trotter share
   estimated from an $M = 32$ spot check (`numpy.random.default_rng(7218)`).
3. Celebrate soberly (prose): agreement on a problem with no closed form, certified by an
   independent numerical experiment, exactly as [§5.8](../05-classical-stat-mech/partition-function.ipynb) certified itself on exact Ising
   results.
4. Deliver the scaling verdict (prose): the grid dies exponentially with dimension; the
   polymer's cost is polynomial; Monte Carlo's dimension-blindness ([§5.7](../05-classical-stat-mech/potentials-legendre-maxwell.ipynb)), quantum
   edition.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    grid_shift < 1e-4,
    "the experiment is trustworthy: grid ED converged under refinement",
    f"E(beta=5) shift {grid_shift:.1e} on doubling resolution and widening the box",
)
validate.close(
    E_q,
    E_ed,
    "certification where no formula exists: PIMC meets grid ED on the energy",
    atol=3 * bE_q,
)
validate.close(
    x2_q,
    x2_ed,
    "and on the thermal width",
    atol=3 * bx_q,
)

## Exercise 8 — (STUDENT/STRETCH) The thermal function of an unsolvable system

$E(T)$ for the quartic oscillator across the quantum–classical bridge. Cite
{eq}`eq-quartic`.

1. Sweep $\beta = 0.5, 1, 2, 3, 5, 8$ at fixed protocol (`pimc_staging`, $M = 64$,
   $L = 32$, `numpy.random.default_rng(7300 + 10*beta)` per point, virial estimator) and
   plot $E(T)$ with bars against the grid-ED curve.
2. Verify the low-$T$ floor against the grid-ED $E_0$ (zero-point energy resolved by coin
   flips) and the high-$T$ climb toward the classical limit, with the classical
   configurational average computed by quadrature (`numpy.trapezoid` over
   $e^{-\beta V}$; equipartition's kinetic $1/2\beta$ stated).
3. Watch the polymer (plot): stored configurations at $\beta = 8$ versus $\beta = 0.5$,
   the loop swelling to the thermal wavelength and shrinking toward a classical point
   (the gem of [§7.20](imaginary-time-quantum-classical.ipynb), animated in stills).
4. Reflect (prose): one algorithm, one dial, and the system walks from quantum zero-point
   to classical equipartition.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    E_floor,
    float(E_g[0]),
    "zero-point energy by coin flips: the low-T floor meets the grid-ED E0",
    atol=3 * bar_floor,
)
validate.check(
    bool(np.all(np.abs(pulls_sweep) < 3.0)),
    "the full thermal function within bars at every temperature",
    "pulls: " + ", ".join(f"{p:+.1f}" for p in pulls_sweep),
)

## Exercise 9 — (Synthesis) Coin flips, trusted

No new computation: state what was computed, and what it cost to believe it.

This notebook computed quantum mechanics with dice, and spent most of its effort on the
harder problem: knowing when to believe the dice. The license came from [§7.20](imaginary-time-quantum-classical.ipynb) — a thermal
particle is a positive-weight classical loop — so the Metropolis of [§5.8](../05-classical-stat-mech/partition-function.ipynb) applied without a
single new idea. Everything else was discipline. Exact finite-$M$ targets to validate
against, because the Gaussian algebra of [§7.20](imaginary-time-quantum-classical.ipynb) made the harmonic polymer solvable at *every*
discretization, not just in the limit. Autocorrelation windows checked for stability, with
a true story about what happens when they are not: a clipped tail, flattered bars, and
honest data wearing a phantom $2.5\sigma$ bias, resolved only because the bridge had been
verified *in isolation* and could not be blamed. Two error budgets never allowed to blur,
one belonging to the sampler and one to the discretization. Estimators chosen for their
variance, not just their expectation. And the slow mode identified by name — the centroid —
because a mover that cannot reach the slow mode is fast only on paper.

The reward was earned honestly. The coth curve arrived a third time, now with error bars:
ladder operators, Gaussian determinants, coin flips, one number. Then a quartic oscillator,
closed to every formula, was pinned within one bar by sampling, in agreement with an
independent numerical experiment, both budgets quoted. There is a particular satisfaction
in the phrase "within one sigma" when both the sigma and the target were fought for. Anyone
can sample; the craft is in the bars. A Monte Carlo result without a validated error
estimate is an anecdote, and this volume does not publish anecdotes.

The movement's machinery is now complete: temperature as geometry ([§7.20](imaginary-time-quantum-classical.ipynb)), geometry as
something a random walk can explore (here). One question remains, and it is the volume's
deepest. Every system in this course was thermal because we coupled it to a bath, explicitly
or by assumption. An isolated quantum system has no bath: its evolution is unitary, its
spectrum discrete, its information never lost. Why, then, does it thermalize at all? That
is [§7.22](eigenstate-thermalization.ipynb).

## Notebook summary

Movement VI's algorithmic notebook: the mapping of [§7.20](imaginary-time-quantum-classical.ipynb), sampled — and a course-length
lesson in trusting stochastic answers.

- **The license** {eq}`eq-pimc-license`: the ring-polymer weight $e^{-S}$ is positive for
  distinguishable particles and bosons, so the Metropolis of [§5.8](../05-classical-stat-mech/partition-function.ipynb) samples quantum mechanics
  verbatim (fermions revoke it: the sign problem, named honestly). The trust protocol:
  validate on exact targets; window-checked bars; budgets separated.
- **Single-bead checkerboard** {eq}`eq-single-bead`: even/odd beads conditionally
  independent (nearest-neighbor action only), so sweeps vectorize; validated against the
  exact $(A^{-1})_{00}$ at $M = 16$ within one bar, with $\tau_{\mathrm{int}} \approx 140$
  sweeps quoted as the price.
- **Diagnostics** {eq}`eq-diagnostics`: self-consistent window $W \ge 8\tau$ with $\tau(c)$
  stability checked; blocking bars at $16\tau$, the block length justified by the measured
  creep. The cautionary tale: a $6\tau$ window manufactured a phantom $2.5\sigma$ bias;
  suspect the bars before the physics.
- **Stiffening** {eq}`eq-stiffening`: springs $\propto M/\beta$; at fixed $\delta$,
  $\tau_{\mathrm{int}}$ climbed $\sim 180 \to \sim 1060$ sweeps from $M = 16 \to 64$
  (gated): local updates drowning near the continuum, the cousin of [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb).
- **Staging** {eq}`eq-staging`: bridge conditionals by precision weighting; verified in
  isolation against $\delta\tau\,k(L-k)/L$ (gated); acceptance near one independent of
  $M$; the centroid lesson measured ($L = M/4$ thrice slower than $M/2$, gated); the
  whole-ring move named.
- **Two budgets** {eq}`eq-two-budgets`: every ladder rung within bars of its own exact
  finite-$M$ value (gated) while the rungs fall onto $\tfrac12\coth(\beta/2)$ as $1/M^2$
  (gated): the three-way coth rendezvous, with bars.
- **Estimators** {eq}`eq-estimators`: primitive and virial both exact for
  $-\partial_\beta\ln Z_M$ (gated at two $M$); sd(primitive) grows $\sim\!\sqrt{M}$ while
  the virial stays flat (gated): virial for production.
- **Certification** {eq}`eq-quartic`: quartic oscillator, no closed form; grid ED
  (refinement-checked) versus PIMC with the full protocol: $E$ and $\langle x^2\rangle$
  within one bar (gated), both budgets quoted; then the full $E(T)$ with the zero-point
  floor resolved (gated) and the classical merge exhibited.

The standing rules issued here: every stochastic number carries a window-checked bar;
statistical and Trotter budgets are quoted separately; the virial estimator is the
production choice; and movers must reach the slow mode, not merely accept often.

## Outlook

- **Eigenstate thermalization** ([§7.22](eigenstate-thermalization.ipynb), the optional capstone): every bath in this volume
  was put in by hand; an isolated quantum system has none, yet its subsystems thermalize.
  Why unitary evolution hides equilibrium inside single eigenstates is the volume's
  closing question.
- **Bosonic exchange as loop reconnection**: permutations connect small loops into long
  ones; sampling reconnections alongside bead moves gives superfluid fractions and
  $\langle N_0\rangle \approx 7$–$10\%$ for helium-4 (Ceperley, *Rev. Mod. Phys.* **67**,
  279 (1995)), meeting the neutron honesty of [§7.17](bose-einstein-condensation.ipynb) from theory.
- **The sign problem**: fermionic permutations carry $(-1)^P$; the variance of signed
  averages grows exponentially in $\beta N$; no general cure is known. The frontier,
  stated without melodrama.
- **Ring-polymer molecular dynamics**: the same polymer run with forces instead of coin
  flips computes nuclear quantum effects in production simulation — the bridge to the
  MMM course, in one line.
- Cross-reference: [§5.7](../05-classical-stat-mech/potentials-legendre-maxwell.ipynb) (dimension-blindness), [§5.8](../05-classical-stat-mech/partition-function.ipynb) (Metropolis and all its disciplines),
  [§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb) (critical slowing, cousin), [§7.5](oscillator-at-temperature.ipynb) (the coth, thrice), [§7.17](bose-einstein-condensation.ipynb) (the condensate honesty
  met), [§7.20](imaginary-time-quantum-classical.ipynb) (the license and the exact targets).

In [ ]:
from ecp.style import footer

footer()